[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_2/L3.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 2 | BÀI 3


---

# **GIẢM CHIỀU DỮ LIỆU: PHÂN TÍCH THÀNH PHẦN CHÍNH**

|  |  |
|:---|:---|
|**Thời gian đọc** |  30 phút|
|**Kiến thức nền** | Học không giám sát, tương quan, không gian riêng (eigenspace)  |
|**Từ khóa** |thành phần chính, hệ số tải (loadings)  |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q pca
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_2/"

*Ở bài đầu tiên, ta đã giới thiệu khái niệm học không giám sát, nghiên cứu các thuật toán phân cụm và xem chúng nhận diện cấu trúc trong một tập dữ liệu như thế nào. Ở bài trước, ta đã áp dụng các khái niệm này với phân cụm phân cấp. Trong bài này, ta chuyển sang một chủ đề khác của học không giám sát: giảm chiều dữ liệu.*

## **1. Giới thiệu**

Các thuật toán giảm chiều nén dữ liệu có nhiều đặc trưng thành dữ liệu có ít đặc trưng hơn, trong khi vẫn giữ lại thông tin quan trọng nhất và loại bỏ thông tin mà ta cho là dư thừa hoặc trùng lặp. Việc này được thực hiện bằng cách chiếu một tập dữ liệu nhiều chiều xuống một không gian có số chiều thấp hơn.

Một quan niệm sai phổ biến trong học máy là càng thêm nhiều đặc trưng vào thuật toán thì mô hình càng chính xác. Điều này đã bị bác bỏ. Người ta đã chỉ ra rằng để cải thiện hiệu suất của các thuật toán học máy, ta nên cân nhắc áp dụng giảm chiều cho các đặc trưng, để chỉ giữ lại những đặc trưng quan trọng và tối ưu cho nhiệm vụ học máy đang quan tâm. Giảm chiều giúp ngăn chặn vấn đề *lời nguyền số chiều* (curse of dimensionality), tức là các thuật toán học máy trở nên kém hiệu quả khi huấn luyện trên dữ liệu có không gian đặc trưng lớn hơn.

Các kỹ thuật giảm chiều phổ biến nhất là
1. Chọn đặc trưng
2. Trích xuất đặc trưng

Giảm chiều được ứng dụng trong
1. Kỹ thuật đặc trưng (feature engineering)
2. Các kỹ thuật giảm nhiễu hoặc lọc
3. Tạo dữ liệu nhân tạo cho các nhiệm vụ học máy.
4. Mô hình hóa tài chính hoặc phân tích rủi ro, bằng cách gộp rất nhiều chỉ số thị trường thành ít thành phần hơn để mô hình hóa hiệu quả

Ta bàn chi tiết về lời nguyền số chiều.

### **1.1 Lời nguyền số chiều**

Các mô hình học máy thường thiếu khớp với dữ liệu có ít đặc trưng và quá khớp với dữ liệu có quá nhiều đặc trưng. Đây là điều ta gọi là lời nguyền số chiều. Nó ngăn nhiều thuật toán học máy đạt kết quả hợp lý vì khiến việc tối ưu thuật toán trở nên bất khả thi.

Khi số chiều của dữ liệu tăng lên, độ khó của việc nhận ra các mẫu hình cũng tăng theo.

### **1.2 Giảm chiều**

Giảm số chiều của một tập dữ liệu sẽ loại bỏ các đặc trưng dư thừa khỏi tập dữ liệu nhưng giữ lại các biến tối ưu để dự báo các biến mục tiêu chính xác hơn.

Giảm chiều cũng làm giảm độ phức tạp của mô hình. Một bước then chốt trong giảm chiều là ma trận hiệp phương sai hoặc ma trận tương quan, giúp kiểm tra tương quan giữa các biến của ta.

### **1.3 Tương quan**

Các biến có tương quan cao với nhau khiến hệ số của các biến đó không ổn định và hoạt động kém khi dùng trong hồi quy. Trong học máy, ta dùng đa cộng tuyến để tìm xem có tồn tại tương quan giữa các biến độc lập hay không.

Sự hiện diện của đa cộng tuyến được kiểm tra bằng Hệ số lạm phát phương sai (VIF), cho bởi
$$\text{VIF} = \frac{1}{1 - R^2}$$

trong đó $R^2 = 1 - \frac{SS_{RES}}{SS_{TOT}} = 1 - \frac{\sum_i (y_i - \hat{y})^2}{\sum_i (y_i - \bar{y})^2}$.

VIF luôn lớn hơn hoặc bằng 1 và được diễn giải như sau:
1. VIF bằng một nghĩa là biến độc lập với các biến khác.
2. VIF từ 1 đến 5 nghĩa là các biến có tương quan vừa phải.
3. VIF lớn hơn 5 nghĩa là các biến có tương quan cao.

Do đó, với một mô hình dự báo đáng mong muốn, ta muốn VIF càng gần một càng tốt.

Giá trị tương quan nằm trong khoảng từ -1 đến +1 và được diễn giải như sau:
1. Giá trị gần +1 cho thấy tương quan dương.
2. Giá trị gần -1 cho thấy tương quan âm.
3. Giá trị gần 0 nghĩa là không tồn tại tương quan giữa hai biến.

Ta có thể trình bày tương quan giữa các biến bằng biểu đồ nhiệt (heatmap) như dưới đây:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets

In [ ]:
# Load Iris Dataset
iris = datasets.load_iris()

# Create a Covariance Matrix
cov_data = np.corrcoef(iris.data.T)
cov_data

In [ ]:
# Plotting the covariance matrix using a heatmap

img = plt.matshow(cov_data, cmap=plt.cm.rainbow)
plt.colorbar(img, ticks=[-1, 0, 1], fraction=0.045)
for x in range(cov_data.shape[0]):
    for y in range(cov_data.shape[1]):
        plt.text(
            x,
            y,
            "%0.2f" % cov_data[x, y],
            size=12,
            color="black",
            ha="center",
            va="center",
        )

plt.show()

Từ biểu đồ nhiệt trên, ta thấy đặc trưng thứ nhất và thứ ba, thứ nhất và thứ tư, thứ ba và thứ tư có tương quan với nhau, trong khi đặc trưng thứ hai có thể coi là độc lập với các đặc trưng còn lại.

Do đó, trong ba đặc trưng tương quan cao, ta có thể giữ một và bỏ hai, vì trong trường hợp này ta coi chúng là thông tin trùng lặp.

Như đã nêu, ta có thể dùng các kỹ thuật sau để giảm chiều.
- Chọn đặc trưng
- Trích xuất đặc trưng

#### **Chọn đặc trưng**
Với kỹ thuật này, ta chọn một tập con trong các biến gốc, tức là chọn những biến biểu diễn tốt hơn các biến khác.

Ta có thể dùng chọn đặc trưng trong một trong các trường hợp sau:
- Khi các đặc trưng không phải là số
- Khi ta cần trích xuất các đặc trưng quan trọng
- Để giữ nguyên các phép đo như ban đầu, không biến đổi.

Ta có thể quan sát việc chọn đặc trưng bằng phương trình vector sau
$$\vec{x} = \pmatrix{x_1\\ x_2\\ \vdots \\ \vdots\\ x_N} → \pmatrix{x_{i1}\\ x_{i2}\\ \vdots \\ x_{ik}}$$
trong đó $k<< N$

Tuy nhiên, chọn đặc trưng đòi hỏi một chiến lược tìm kiếm để chọn các đặc trưng tốt nhất, và điều này có thể là thách thức với mô hình học không giám sát. Việc thiếu thông tin về nhóm khiến khó đo được đặc trưng nào sẽ cải thiện hiệu suất của mô hình học máy. Các kỹ thuật chọn đặc trưng phổ biến khi không có thông tin về nhóm là chọn các biến có phương sai cao hơn.

#### **Trích xuất đặc trưng**

Với phương pháp này, ta biến đổi các đặc trưng thô thành các đặc trưng mới bằng cách ánh xạ từ các đặc trưng gốc
$$\vec{x} = \pmatrix{x_1\\ x_2\\ \vdots \\ \vdots\\ x_N} \xrightarrow{f(\vec{x})} \pmatrix{y_1\\ y_2\\ \vdots \\ y_k}$$
trong đó $k<< N$.

Phép biến đổi có thể là tuyến tính hoặc phi tuyến.

Các kỹ thuật trích xuất đặc trưng tuyến tính được liệt kê dưới đây:
1. Phân tích thành phần chính (PCA)
2. Phân tích biệt thức tuyến tính (LDA)

Trong bài này, ta tập trung vào PCA.

## **2. Phân tích thành phần chính (PCA)**

PCA là một kỹ thuật hữu ích để giảm số chiều của tập đặc trưng trong khi vẫn giữ thông tin từ các đặc trưng gốc.
Nó làm được điều này qua việc trích xuất đặc trưng, tức là các đặc trưng mới được tạo ra bằng cách lấy tổ hợp tuyến tính của các đặc trưng gốc.
Khi áp dụng PCA cho một tập dữ liệu, ta có thể học được các cách biểu diễn khác nhau của dữ liệu, giải thích các điểm dữ liệu gốc.

Phép biến đổi tuyến tính của các đặc trưng có thể biểu diễn bằng phương trình $$y = WX$$ trong đó $W$ là các trọng số, $X$ là các đặc trưng gốc và $y$ là các thành phần chính.

Phép biến đổi tuyến tính có một thách thức là nó làm các đặc trưng sau biến đổi khó diễn giải hơn, do đó không hữu ích cho phân tích tiếp theo.

Các thành phần chính, $y_i$, giải thích một tỷ lệ phần trăm cao hơn trong biến thiên của dữ liệu gốc, và các trọng số $W_{ij}$ cho thấy đóng góp của các đặc trưng gốc vào các thành phần chính tạo ra.

Xét ví dụ với hai biến; $$y_i = w_{i,1} x_1 + w_{i2} x_2$$

Ta gọi các trọng số $(w_{i,1}, w_{i,2})$ là **hệ số tải** (loadings) của thành phần. Thành phần chính thứ nhất, $y_1$, giải thích phần lớn biến thiên trong tập dữ liệu của ta, còn $y_2$, trực giao với $y_1$, giải thích phần lớn biến thiên còn lại. Nếu có nhiều đặc trưng hơn, các thành phần chính bổ sung cũng sẽ trực giao với nhau.

Các biến mới tìm được có thể được kết hợp tuyến tính để thu lại các đặc trưng dữ liệu gốc. Các biến mới này gọi là các thành phần chính (PC) và các PC trực giao (độc lập) với nhau.

Phân tích thành phần chính có thể dùng theo các cách sau:
1. Trực quan hóa một tập dữ liệu nhiều chiều thành các biểu đồ có số chiều thấp hơn.
2. Loại bỏ nhiễu khỏi dữ liệu.
3. Tìm cấu trúc trong một tập dữ liệu.
4. Giảm quá khớp trong dữ liệu.

Về mặt toán học, ta có thể khái quát các mục tiêu của PCA như sau:
- Tìm một cơ sở trực chuẩn cho tập dữ liệu.
- Sắp xếp các chiều theo thứ tự tầm quan trọng.
- Loại bỏ các chiều có ý nghĩa thấp hơn.
- Đảm bảo các biến mới không tương quan với nhau.

### **2.1 Tính các thành phần chính và hệ số tải**

Việc tính các thành phần chính dùng ma trận tương quan hoặc ma trận hiệp phương sai của dữ liệu, và chỉ áp dụng với các đặc trưng là số. Sau đây là các bước thực hiện giảm chiều.

1. Chuẩn hóa: bước đầu tiên trong quy trình PCA là chuẩn hóa tập dữ liệu. Điều này cần thiết vì PCA nhạy với những tập dữ liệu có các đặc trưng với khoảng giá trị khác nhau hoặc có giá trị ngoại lai. Bước này đảm bảo mỗi đặc trưng đóng góp như nhau. Chuẩn hóa được thực hiện bằng cách trừ trung bình của mỗi đặc trưng rồi chia cho độ lệch chuẩn của nó. Đặc trưng đã chuẩn hóa sẽ có trung bình bằng không và độ lệch chuẩn bằng một (độ dài đơn vị bằng một).

2. Đưa dữ liệu về gốc tọa độ: ta tiếp tục tính trung bình của mỗi đặc trưng và tìm tâm của dữ liệu. Ta dịch chuyển dữ liệu sao cho tâm nằm ở gốc tọa độ mà không làm thay đổi khoảng cách giữa các điểm dữ liệu.

3. Thành phần chính thứ nhất (PC1) được tính bằng cách tìm tổ hợp tuyến tính của các đặc trưng làm cực đại tổng biến thiên trong dữ liệu
$$Y_1 = a_{11} X_1 + a_{12} X_2 + \cdots + a_{1N} X_N$$ Hệ quả là ta có thể làm phương sai của $Y_1$ lớn tùy ý. Để ngăn điều đó xảy ra, tổng bình phương của các trọng số được đặt bằng 1, tức là
$$a_{11}^2 + a_{12}^2 +\cdots a_{1N}^2 = 1$$

4. Ta lặp lại quy trình này bằng cách tìm thành phần chính (PC2) đi qua gốc tọa độ, trực giao (tức không tương quan) với PC1 và giải thích phương sai cao tiếp theo.
$$Y_2 = a_{21} X_1 + a_{22} X_2 + \cdots + a_{2N} X_N$$
5. Quá trình này tiếp tục cho đến khi ta có số thành phần chính bằng số đặc trưng gốc. Khi đã có tất cả các thành phần chính, tổng phương sai của các thành phần chính sẽ bằng tổng phương sai của các đặc trưng gốc, nhờ đó ta chắc chắn không có thông tin nào bị mất trong quá trình biến đổi. Như thấy ở các bước trên, PCA xoay các mẫu dữ liệu theo hướng của các trục, rồi dịch chuyển dữ liệu để trung bình bằng 0, nhưng quan trọng nhất là đảm bảo không mất thông tin trong quá trình đó.

Ở dạng ma trận, PCA có thể viết là $$Y= XA$$

Các hàng của ma trận $A$ là các vector riêng và chúng xác định các thành phần chính được định hướng thế nào so với các đặc trưng gốc. Thành phần của các vector riêng là các trọng số $a_{ij}$ và được gọi là hệ số tải. Hệ số tải cho ta biết mỗi đặc trưng đóng góp thế nào vào một thành phần chính.

Hệ số tải nằm trong khoảng từ $-1$ đến $1$; hệ số tải lớn (gần $1$ hoặc $-1$) cho thấy có mối quan hệ mạnh giữa một đặc trưng cụ thể và thành phần chính, còn hệ số tải gần không cho thấy mối quan hệ yếu hơn. Dấu của hệ số tải cho biết đặc trưng tương quan dương hay âm với thành phần chính.

Cho ma trận $A$ và ma trận phương sai–hiệp phương sai của dữ liệu gốc, $S_X$, thì ma trận phương sai–hiệp phương sai của các thành phần chính thu được sẽ là $$S_Y = A S_X A^T$$

$S_Y$ là ma trận đường chéo có các giá trị là trị riêng (hay phương sai được giải thích) bởi các thành phần chính tương ứng. PCA thực hiện giảm chiều bằng cách chiếu dữ liệu lên không gian thành phần chính rồi sắp thứ tự các PC theo hướng có phương sai lớn nhất trong tập dữ liệu. Điều này đảm bảo các PC không tương quan với nhau và tạo thành một tập cơ sở trực giao.
Do đó các trị riêng được sắp xếp giảm đơn điệu từ lớn nhất đến nhỏ nhất. Ta dùng **biểu đồ scree** để trực quan hóa tốc độ phương sai được giải thích giảm đi khi ta thêm nhiều thành phần chính.

Một biểu đồ scree lý tưởng sẽ uốn đều và giống biểu đồ khuỷu tay, trong đó các PC không đáng kể sẽ phẳng dần, tạo ra một điểm cắt rõ ràng giúp ta dễ chọn số PC tối ưu. Trường hợp biểu đồ scree không lý tưởng, ta có thể dùng các quy tắc dưới đây để chọn PC.
1. Áp dụng quy tắc Kaiser, tức là các PC phải có trị riêng ít nhất bằng 1.
2. Các PC được chọn phải đóng góp ít nhất $80\%$ phương sai.

Nếu thu được nhiều hơn 3 PC, ta nên cân nhắc dùng các kỹ thuật giảm chiều khác vì PCA sẽ không phải cách tốt nhất để trực quan hóa dữ liệu. Trong hệ tọa độ mới, vị trí của các thành phần chính được gọi là điểm số (scores).

Dưới đây là phần ôn lại các không gian riêng mà ta đã học ở trên.

### **2.2 Toán học đằng sau PCA**

Dưới đây ta ôn lại khái niệm không gian riêng;

Cho **A** là ma trận $n×n$. Các trị riêng của **A** là nghiệm của phương trình đặc trưng
$$\text{determinant}(\textbf{A} - λ \textbf{I}) = |(\textbf{A} - λ \textbf{I})| = 0$$
trong đó $\textbf{I}$ là ma trận đơn vị $n× n$.

Nếu $λ$ là một trị riêng của $\textbf{A}$ thì tồn tại một vector $\vec{x}$ sao cho
$$\textbf{A} \vec{x} = λ \vec{x}$$

Vector $\vec{x}$ này gọi là vector riêng của ma trận $\textbf{A}$ ứng với trị riêng $λ$ và nó không duy nhất.

Xét ma trận $\textbf{A}$ cỡ $2×2$ có các vector riêng $\vec{x_1}, \vec{x_2}$ và các trị riêng $λ_1, λ_2$. Từ định nghĩa vector riêng, ta có
$$\textbf{A}\vec{x_1} = λ_1\vec{x_1}\\\textbf{A}\vec{x_2} = λ_2\vec{x_2}$$
có thể viết thành:
$$\textbf{A}[\vec{x_1}\hspace{3mm}\vec{x_2}] = [\vec{x_1}\hspace{3mm}\vec{x_2}]\pmatrix{λ_1 & 0\\ 0 & λ_2}$$.

Đặt $𝚽 = [\vec{x_1}\hspace{3mm}\vec{x_2}]$ và $Λ = \pmatrix{λ_1 & 0\\ 0 & λ_2}$. Khi đó: $$\textbf{A}𝚽 = 𝚽 Λ$$

Chuẩn hóa các vector riêng sao cho chúng trực giao, ta có;
$$𝚽 𝚽^T = 𝚽^T 𝚽 = \textbf{I}$$
suy ra
$$𝚽^T \textbf{A}𝚽 = Λ$$
và
$$\textbf{A}=𝚽 Λ 𝚽^T$$.

Điều này áp dụng vào quy trình PCA như thế nào?

Cho $S$ là ma trận hiệp phương sai (tương quan) cỡ $m×m$, tồn tại một ma trận trực giao $m\times m$, $𝚽$, sao cho các vector riêng của S là các cột của $𝚽$ và các trị riêng của $S$ là đường chéo của ma trận $Λ$ sao cho:
$$𝚽^T S 𝚽 = \Lambda$$
Các vector riêng của ma trận hiệp phương sai (tương quan) cho ta các vector chỉ hướng $Φ_1, Φ_2$. Ta có thể tạo một ma trận biến đổi từ các vector riêng để chuyển dữ liệu từ hệ trục $[x_1, x_2]$ sang hệ trục $[Φ_1, Φ_2]$ bằng phương trình
$$p_{Φ} = (p_x - \mu_x)\cdot Φ$$
với $p_x$ là một điểm bất kỳ trong hệ trục $[x_1, x_2]$ và $\mu_x$ là trung bình dữ liệu.

### **2.3 Triển khai PCA**

Ta bắt đầu bằng cách cài đặt thư viện PCA.

Sau đó, nạp các thư viện cần cho **phần** này.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pca import pca

Bước tiếp theo là tạo một tập dữ liệu để áp dụng thuật toán PCA.

In [ ]:
# Dataset
from sklearn.datasets import load_wine

# Load dataset
data = load_wine()
X = data.data
y = data.target
labels = data.feature_names
# Make dataframe
df = pd.DataFrame(data=X, columns=labels, index=y)

In [ ]:
X.shape

In [ ]:
# Initialize with normalization
model = pca(normalize=True, n_components=None)

In [ ]:
# Fit transform with dataframe
out = model.fit_transform(df)

Biểu đồ scree dưới đây giúp ta thấy phần trăm phương sai mà tất cả các PC nắm bắt được. Ta có thể cần xem xét các kỹ thuật giảm đặc trưng khác vì phải mất 5 biến mới đạt tỷ lệ phương sai tích lũy $80\%$.

In [ ]:
# Scree plot with explained variance
print("\033[1m" + "Fig. 1: Scree Plot" + "\033[0m")
model.plot()

Bây giờ ta có thể xem hệ số tải trong đoạn mã dưới đây. Như đã nêu ở các phần trước, hệ số tải giúp ta thấy các đặc trưng đóng góp thế nào vào các PC và các nhóm được phân chia ra sao.

In [ ]:
# Top performing features
model.results["topfeat"]

Bây giờ ta có thể trực quan hóa biểu đồ phân tán của các điểm số như dưới đây.

In [ ]:
# Make plots

# Scatter plot
print("\033[1m" + "Fig. 2: Scatter plot of the Scores" + "\033[0m")
fig, ax = model.scatter(figsize=(10, 6))

Từ biểu đồ phân tán, ta vẽ biplot và nhận thấy 2 PC đầu tiên có thể dùng đủ tốt để tách 3 nhóm đầu tiên.

In [ ]:
# Biplot
print("\033[1m" + "Fig. 3: Biplot" + "\033[0m")
fig, ax = model.biplot(figsize=(10, 6))

Để có hình biplot rõ ràng, không kèm các điểm số, xem bên dưới. Lưu ý góc giữa hai đặc trưng càng nhỏ thì chúng càng đóng góp nhiều vào một thành phần chính.

In [ ]:
print("\033[1m" + "Fig. 3: Biplot without the scores" + "\033[0m")
fig, ax = model.biplot(cmap=None, label=False, legend=False, figsize=(10, 6))

## **3. Kết luận**

Trong bài này, ta đã giới thiệu khái niệm Phân tích thành phần chính, và ở bài tiếp theo, ta sẽ áp dụng kiến thức này lên dữ liệu lợi suất (yield).

**Tài liệu tham khảo**

- Abdi, Hervé, and Lynne J. Williams. "Principal Component Analysis." *Wiley Interdisciplinary Reviews: Computational Statistics*, vol. 2. no. 4, 2010, pp. 433-459.

- Ringnér, Markus. "What is Principal Component Analysis?" *Nature Biotechnology*, vol. 26, no. 3, 2008, pp. 303-304.